In [0]:
from pyspark.sql import functions as F

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA default")
P = "wdatt_movie_"

checks = {
  f"{P}bronze_movies": None,
  f"{P}bronze_links": None,
  f"{P}bronze_tags": None,
  f"{P}bronze_scraped_metadata": None,
  f"{P}bronze_ratings_top500": None,
  f"{P}silver_movie_master": None,
  f"{P}gold_dim_movies_enriched": 500,
  f"{P}gold_fact_ratings": None,
  f"{P}gold_dim_users": None,
  f"{P}serving_user_recommendations_enriched": None,
}

for t, expected in checks.items():
    c = spark.table(t).count()
    msg = f"{t}: {c:,}"
    if expected is not None:
        msg += f" (expected {expected})"
        assert c == expected, msg
    print(msg)

# quick sanity on rec ranks
rec = spark.table(f"{P}serving_user_recommendations_enriched")
bad = rec.where((F.col("rec_rank") < 1) | (F.col("rec_rank") > 10)).count()
print("bad rec_rank rows:", bad)
assert bad == 0
print("✅ END-TO-END validated")
